# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ARS-0/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup\n\n*Finds the data and the receipts (the metrics JSONs from Weeks 4-7), whichever way the folder is capitalised.*

In [1]:
# Run all takes under a minute. This notebook does NOT retrain anything: it re-derives the paper's key numbers from the raw CSV
# and checks them against the committed "receipts" (the metrics JSONs from Weeks 4-7), then collects the files the paper embeds.
import os, sys, subprocess, json, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ARS-0/FlyRank-ML-Internship"
REPO_DIR = "FlyRank-ML-Internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"

# The metrics JSONs may sit in work/outputs or work/Outputs depending on how they were uploaded: find them either way.
def find_dir(parent, name):
    for p in os.listdir(parent):
        if p.lower() == name.lower() and os.path.isdir(os.path.join(parent, p)):
            return os.path.join(parent, p)
    os.makedirs(os.path.join(parent, name), exist_ok=True)
    return os.path.join(parent, name)
OUT = find_dir("work", "outputs")
FIG = find_dir("work", "figures")
def receipt(name):
    path = os.path.join(OUT, name)
    assert os.path.exists(path), f"{name} not found in {OUT}. Run the Week 4-7 notebooks (or commit their metrics JSONs) first."
    return json.load(open(path))
w04, w05, w06, w07 = (receipt(f) for f in ["w04_baseline_metrics.json", "w05_model_metrics.json", "w06_audit_metrics.json", "w07_playbook_metrics.json"])
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Receipts found in:", OUT)
print("CSV rows, columns:", df.shape, "| clients:", df["client_id"].nunique())

Receipts found in: work/Outputs
CSV rows, columns: (30000, 44) | clients: 32


## 1. Question

*The research question and the decision it supports.*

In [2]:
# The decision: which pages should a person read first?  Unit = one page. Output = ranked score + one reason code. Action = a person reads the page.
print(f"Pages in the file: {len(df):,} | pages in the review queue: {w07['queue_pages']:,} | pages with an observed outcome: {w07['validated_pages']:,}")
print("Question: of pages already ranking in search, which will get noticeably fewer clicks than same-position peers in the next 30 days?")

Pages in the file: 30,000 | pages in the review queue: 15,441 | pages with an observed outcome: 9,102
Question: of pages already ranking in search, which will get noticeably fewer clicks than same-position peers in the next 30 days?


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [3]:
# Re-derive the paper's population counts straight from the raw CSV (no receipts used here).
df["pre_impr"] = df["impressions_90d"] - df["impressions_last_30d"]       # feature window = days 31-90 back
df["pre_clk"]  = df["clicks_90d"] - df["clicks_last_30d"]
df["early_impr"] = df["pre_impr"] - df["impressions_prev_30d"]
assert (df[["pre_impr", "pre_clk", "early_impr"]] >= 0).all().all()

n_gate100 = int((df["impressions_90d"] >= 100).sum())
queue = df[(df["pre_impr"] >= 500) & (df["avg_position"] > 0)]
valid = queue[queue["impressions_last_30d"] >= 500]
no_pos = int((df["avg_position"] == 0).sum())
print(f"Pages in file: {len(df):,} | clients: {df['client_id'].nunique()} | rows with no position data (avg_position = 0): {no_pos:,}")
print(f"Pages with >= 100 impressions in 90 days (Week-4 screen): {n_gate100:,}   [receipt: {w04['rows_scored_impressions_ge_100']:,}]")
print(f"Queue population (>= 500 impressions before the outcome window, real position): {len(queue):,}, {queue['client_id'].nunique()} clients   [receipt: {w07['queue_pages']:,}]")
print(f"Validated population (also >= 500 impressions in the outcome window): {len(valid):,}, {valid['client_id'].nunique()} clients   [receipt: {w07['validated_pages']:,}]")
print(f"Dropped for low outcome-window impressions: {1 - len(valid)/len(queue):.0%}")
assert n_gate100 == w04["rows_scored_impressions_ge_100"] and len(queue) == w07["queue_pages"] and len(valid) == w07["validated_pages"]
print("\nCounts match the committed receipts.")

Pages in file: 30,000 | clients: 32 | rows with no position data (avg_position = 0): 1,205
Pages with >= 100 impressions in 90 days (Week-4 screen): 22,006   [receipt: 22,006]
Queue population (>= 500 impressions before the outcome window, real position): 15,441, 28 clients   [receipt: 15,441]
Validated population (also >= 500 impressions in the outcome window): 9,102, 28 clients   [receipt: 9,102]
Dropped for low outcome-window impressions: 41%

Counts match the committed receipts.


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [4]:
# Re-derive the label and its base rate from raw data, then compare with the receipt.
bench = valid.groupby("position_tier").apply(lambda x: x["clicks_last_30d"].sum() / x["impressions_last_30d"].sum())
y = (valid["clicks_last_30d"] <= 0.5 * valid["position_tier"].map(bench) * valid["impressions_last_30d"]).astype(int)
print("Tier yardstick, pooled CTR in the outcome window (%):", (bench * 100).round(3).to_dict())
print(f"Label base rate: {y.mean():.4f}   [receipt: {w05['base_rate']}]")
assert abs(y.mean() - w05["base_rate"]) < 1e-3

feature_groups = {
    "feature window (days 31-90 back)": ["log_pre_impr", "log_pre_clk", "pre_ctr_pct", "log_prev30_impr", "prev30_ctr_pct"],
    "static page facts (14 columns)": ["log_volume", "competition_f", "cpc_f", "word_count_f", "has_word_count", "has_volume", "log_age", "content_type (one-hot)", "main_intent (one-hot)"],
    "90-day position columns that OVERLAP the outcome (kept, tested)": ["avg_position", "position_tier (one-hot)"],
}
for k, v in feature_groups.items(): print(f"- {k}: {', '.join(v)}")
print("\nNever used as features: trend_direction, trend_pct, any *_last_30d column, 90-day totals, ctr, days_since_last_update, IDs.")
print("Validation:", w05["cv"], "| seed:", w05["seed"], "| Colab scikit-learn used for the committed notebooks was 1.6.1; JSON receipts record 1.8.0 / 1.9.1")

Tier yardstick, pooled CTR in the outcome window (%): {'deep': 0.024, 'page_1': 0.361, 'page_3_5': 0.22, 'striking': 0.429, 'top_3': 0.57}
Label base rate: 0.3606   [receipt: 0.3606]
- feature window (days 31-90 back): log_pre_impr, log_pre_clk, pre_ctr_pct, log_prev30_impr, prev30_ctr_pct
- static page facts (14 columns): log_volume, competition_f, cpc_f, word_count_f, has_word_count, has_volume, log_age, content_type (one-hot), main_intent (one-hot)
- 90-day position columns that OVERLAP the outcome (kept, tested): avg_position, position_tier (one-hot)

Never used as features: trend_direction, trend_pct, any *_last_30d column, 90-day totals, ctr, days_since_last_update, IDs.
Validation: GroupKFold(5) by client_id | seed: 42 | Colab scikit-learn used for the committed notebooks was 1.6.1; JSON receipts record 1.8.0 / 1.9.1


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [5]:
mvb = pd.DataFrame(w05["model_vs_baseline"]).T[["precision@50", "precision@100", "precision@200", "AUC"]]
print(f"MODEL vs BASELINE, split by client, 5 folds (base rate {w05['base_rate']:.3f} overall):")
print(mvb.round(3).to_string())

ba = pd.DataFrame(w06["before_after_precision_at_50"]).T
print("\nBEFORE / AFTER the honest split (precision@50):")
print(ba.round(3).to_string())
print("\n20 random client holdouts: Logistic Regression beats the rule in", w06["repeated_client_holdouts_20"]["LogReg_beats_rule"], "and Random Forest in", w06["repeated_client_holdouts_20"]["RF_beats_rule"], "of 20")

MODEL vs BASELINE, split by client, 5 folds (base rate 0.361 overall):
                           precision@50  precision@100  precision@200    AUC
Baseline rule (W04 logic)         0.704          0.636          0.588  0.738
Decision tree (depth 3)           0.644          0.680          0.666  0.790
Logistic Regression               0.800          0.764          0.742  0.823
Random Forest                     0.816          0.818          0.752  0.834

BEFORE / AFTER the honest split (precision@50):
                                                   Random Forest  Logistic Regression  Rule baseline
A  BEFORE: 90-day features + random split                  0.996                0.984          0.724
B  window-only features + random split                     0.928                0.880          0.724
C  AFTER:  window-only features + split by client          0.816                0.800          0.704
D  90-day features + split by client                       0.972                0.964      

## 5. Limitations

*What this work cannot claim.*

In [6]:
# Facts behind the limitations, re-derived from raw data.
comp = df[df["content_type"] == "comparison article"]
print(f"Comparison articles in the file: {len(comp):,}, all from {comp['client_id'].nunique()} client(s)  [receipt: {w06['comparison_articles_clients']}]")
print("Content types among the validated pages:", valid["content_type"].value_counts().to_dict())
print(f"Biggest client's share of the validated pages: {valid['client_id'].value_counts(normalize=True).iloc[0]:.0%}")
print(f"Biggest client's share of the top-100 queue: {w07['top100_biggest_client_share']:.0%} (before the 25-pages-per-client review cap)")
print("Staleness (days since update) vs fell-next-month rate [pages, rate]:", w07["staleness_fell_rate"])

Comparison articles in the file: 697, all from 1 client(s)  [receipt: 1]
Content types among the validated pages: {'keyword article': 9056, 'feedly article': 38, 'comparison article': 8}
Biggest client's share of the validated pages: 35%
Biggest client's share of the top-100 queue: 54% (before the 25-pages-per-client review cap)
Staleness (days since update) vs fell-next-month rate [pages, rate]: {'0-30': [6513, 0.606], '31-90': [35, 0.543], '91-180': [4560, 0.611], '181+': [11, 1.0]}


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [7]:
codes = pd.DataFrame({"pages_in_queue": w07["reason_code_counts"],
                      "share_under_captured_clicks": w07["reason_code_share_under_capturing"],
                      "share_impressions_fell_20pct": w07["reason_code_share_impressions_fell_20"]}).sort_values("share_under_captured_clicks", ascending=False)
print(f"Reason codes (base rate for under-capturing {w07['base_rate']:.3f}; impressions-fell rate is ~0.61 overall):")
print(codes.round(3).to_string())
print("\nTop-50 realized missed clicks per page: playbook", w07["mean_realized_missed_clicks_top50_playbook"], "| rule only", w07["rule_only_top50_mean_missed_clicks"],
      "| average page", w07["mean_realized_missed_clicks_all_validated"])
print("Momentum (pages, share that fell next month):", w07["fade_fell_rate"])

Reason codes (base rate for under-capturing 0.361; impressions-fell rate is ~0.61 overall):
                      pages_in_queue  share_under_captured_clicks  share_impressions_fell_20pct
DATA_CHECK_NO_CLICKS             452                        0.742                         0.811
CTR_BELOW_TIER_PEERS            4804                        0.655                         0.669
POSITION_LIMITED                1981                        0.382                         0.620
FADING_IMPRESSIONS              4492                        0.249                         0.670
NO_FLAG                         3712                        0.173                         0.470

Top-50 realized missed clicks per page: playbook 119.81 | rule only 122.96 | average page 5.16
Momentum (pages, share that fell next month): {'fell > 26%': [4321, 0.701], 'fell 10-26%': [1810, 0.581], 'about flat': [1722, 0.532], 'rose > 10%': [3203, 0.54], 'too little history (<100 early impressions)': [63, 0.571]}


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [8]:
# New figure for the paper: what the honest validation changed. Built from the Week-6 receipt.
import textwrap
labels = {"A  BEFORE: 90-day features + random split": "Before: 90-day inputs,\nrandom split",
          "B  window-only features + random split": "Overlap removed,\nrandom split",
          "C  AFTER:  window-only features + split by client": "After: overlap removed,\nsplit by client",
          "D  90-day features + split by client": "Overlap kept,\nsplit by client"}
order = list(labels)
series = {"Rule baseline": ("#7f7f7f", "Rule baseline"), "Logistic Regression": ("#0072B2", "Logistic Regression"), "Random Forest": ("#E69F00", "Random Forest")}
fig, ax = plt.subplots(figsize=(7.2, 4.4))
w = 0.26
for j, (key, (col, name)) in enumerate(series.items()):
    vals = [w06["before_after_precision_at_50"][k][key] for k in order]
    bars = ax.bar(np.arange(len(order)) + (j - 1) * w, vals, width=w, color=col, label=name)
    for b, v in zip(bars, vals): ax.text(b.get_x() + b.get_width() / 2, v + 0.012, f"{v:.2f}", ha="center", fontsize=7)
ax.axhline(w06["base_rate"], color="black", ls="--", lw=1)
ax.text(-0.45, w06["base_rate"] + 0.015, f"base rate {w06['base_rate']:.2f}", fontsize=8, bbox=dict(fc="white", ec="none", pad=1))
ax.axvspan(1.5, 2.5, color="#009E73", alpha=0.10, lw=0)
ax.text(2, 1.045, "my reported design", ha="center", fontsize=8, color="#006b4d")
ax.set_xticks(range(len(order))); ax.set_xticklabels([labels[k] for k in order], fontsize=8)
ax.set_ylim(0, 1.12); ax.set_ylabel("precision@50 (share of the 50 picks that\nreally under-captured)")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=3, frameon=False)
ax.set_title("The same models under four ways of testing", fontsize=10)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "w08_validation_before_after.png"), dpi=160); plt.close(fig)

# Collect everything the deployed page embeds into docs/ (relative paths, no absolute links).
os.makedirs("docs/img", exist_ok=True); os.makedirs("docs/receipts", exist_ok=True)
paper_figs = ["w08_validation_before_after.png", "w07_topk_precision.png", "w07_reason_code_evidence.png", "w07_fade_momentum.png"]
for f in paper_figs:
    src = os.path.join(FIG, f); assert os.path.exists(src), f"missing figure {f}"
    shutil.copy(src, os.path.join("docs/img", f))
for f in ["w04_baseline_metrics.json", "w05_model_metrics.json", "w06_audit_metrics.json", "w07_playbook_metrics.json"]:
    shutil.copy(os.path.join(OUT, f), os.path.join("docs/receipts", f))
print("Figures in docs/img:", sorted(os.listdir("docs/img")))
print("Receipts in docs/receipts:", sorted(os.listdir("docs/receipts")))
print("docs/index.html present:", os.path.exists("docs/index.html"))

Figures in docs/img: ['w07_fade_momentum.png', 'w07_reason_code_evidence.png', 'w07_topk_precision.png', 'w08_validation_before_after.png']
Receipts in docs/receipts: ['w04_baseline_metrics.json', 'w05_model_metrics.json', 'w06_audit_metrics.json', 'w07_playbook_metrics.json']
docs/index.html present: True


## ML-12 — Sharing the work

### 5-minute demo outline

1. **0:00-0:45, the problem.** An editor has about 15,000 pages that show up in search and time to read 30 a week. Which 30?
2. **0:45-1:45, the data and the test.** One picture: older days are the inputs, the last 30 days are the outcome. Pages are scored by a model that never saw their client.
3. **1:45-2:45, the result that matters.** The honest-validation chart. Same models: 80% on new clients, 98-100% with a random split and inputs that overlap the outcome. The gap was my own leak.
4. **2:45-3:45, the queue.** Reason codes, what each one turned out to mean, and the person-reads-the-page rules.
5. **3:45-4:30, what I got wrong.** The comparison-article finding came from one client. I reported it as unproven. Page age showed no effect; fading did.
6. **4:30-5:00, limits and next step.** One snapshot, observed and directional only. The next step is a small test with an untouched comparison group.

### Social post cut

> I tested a page-review queue on search data from clients it had never seen. Its top 50 picks were right about 80% of the time (a hand-written rule got 70%, and roughly 1 in 3 pages qualifies). With a random split and inputs that overlapped the outcome, the same model looked 99% accurate. That gap was my own leak, and I show it in the write-up. Charts, method and code: https://ars-0.github.io/FlyRank-ML-Internship/

### Three sentences for an employer

I built and tested a ranked review queue on FlyRank's anonymized search data (30,000 pages from 32 clients), comparing a plain rule with two models on clients they had never seen. The top 50 picks were right about 80% of the time against 70% for the rule, and when a pattern I had reported (comparison articles underperform) turned out to come from a single client, I wrote it up as unproven instead of keeping the finding. Everything is public and checkable, including the mistakes I fixed: https://ars-0.github.io/FlyRank-ML-Internship/

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
